# 004b — Memory-safe spatial held-out top-m expansion

This is the replacement for the original `004b_expand_spatial_heldout_topm_fullgrid.ipynb`.

It fixes the original notebook's two main resource problems:

1. the across-condition fit is **not reloaded inside every outer split/direction**;
   it is loaded once for the current `K × condition`, then reused for all 40
   outer split/direction evaluations;
2. decoding repetitions are processed sequentially and top-m reconstructions
   are accumulated in one reusable buffer.

The statistical analysis is unchanged: existing held-out rankings, exact stored
outer seeds, A→B/B→A evaluation, 20 outer splits, 20 repeated two-fold decoding
partitions, and the same correlation-based timepoint decoder.


In [ ]:
from pathlib import Path
import shutil, sys, gc
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from IPython.display import display

REPO = Path.cwd()
sys.path.insert(0, str(REPO / "src"))

ANALYSIS_TYPE = "spatial"
FIT_SCOPE = "across"
OUTPUT_ROOT = Path(".")
FIT_DIR = Path("msaa_flexible_outputs_npz")

SOURCE_DIR = Path("msaa_heldout_topm_outputs_spatial_across")
OUTPUT_DIR = Path("msaa_heldout_topm_outputs_spatial_across_fullgrid")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_RUN = SOURCE_DIR / "heldout_topm_crossfit_runs.csv"
SOURCE_RANK = SOURCE_DIR / "heldout_per_archetype_ranking_runs.csv"
RUN_CSV = OUTPUT_DIR / "heldout_topm_crossfit_runs.csv"
RANK_CSV = OUTPUT_DIR / "heldout_per_archetype_ranking_runs.csv"
SUMMARY_CSV = OUTPUT_DIR / "heldout_topm_summary.csv"

TOP_M_VALUES = [1,2,3,5,7,10,15,20,25,30,40,50,75,100]
COMPACT_SPATIAL_K = [70,105,140,175,189,252]
NREPS = 20
RNG_SEED = 42
EPS = 1e-8
DTYPE = np.float64

if not SOURCE_RANK.exists():
    raise FileNotFoundError(SOURCE_RANK)
if not RANK_CSV.exists():
    shutil.copy2(SOURCE_RANK, RANK_CSV)
if SOURCE_RUN.exists() and not RUN_CSV.exists():
    shutil.copy2(SOURCE_RUN, RUN_CSV)

rank_df = pd.read_csv(RANK_CSV)
run_df = pd.read_csv(RUN_CSV) if RUN_CSV.exists() and RUN_CSV.stat().st_size else pd.DataFrame()

print("Ranking rows:", len(rank_df), flush=True)
print("Existing run rows:", len(run_df), flush=True)
print("K requested:", COMPACT_SPATIAL_K, flush=True)
print("Missing ranking K:", sorted(set(COMPACT_SPATIAL_K)-set(rank_df["K"].astype(int))), flush=True)


In [ ]:

def _normalize_labels(labels):
    out = []
    for x in np.asarray(labels).reshape(-1):
        if isinstance(x, bytes):
            out.append(x.decode("utf-8"))
        else:
            out.append(str(x))
    return np.asarray(out, dtype=object)


def load_condition_subjects(K, condition):
    """
    Load the across-condition fit exactly once for this K x condition,
    retain only the requested condition's subjects, then release the
    other 72 subject-condition objects before entering the outer loop.
    """
    source = FIT_DIR / f"spatialAA_across_acrossCond_K{int(K)}.npz"
    if not source.exists():
        raise FileNotFoundError(source)

    with np.load(source, allow_pickle=True) as data:
        labels = _normalize_labels(data["condition_labels_str"])
        idx = np.where(labels == str(condition))[0]

        if len(idx) == 0:
            raise ValueError(
                f"No {condition!r} subjects in {source.name}; "
                f"labels={sorted(set(labels.tolist()))}"
            )

        all_subjects = data["results_subj"].tolist()
        subjects = [all_subjects[int(i)] for i in idx]

        # Release references to the other conditions before the expensive loop.
        del all_subjects

    del labels, idx
    gc.collect()
    return subjects, source


In [ ]:
def get_xval_assignments(n, rng):
    ids = np.zeros(n, dtype=int)
    groupsize = int(np.ceil(n / 2))
    ids[groupsize:] = 1
    rng.shuffle(ids)
    return ids


def decoder_metrics(C):
    T = C.shape[0]
    t = np.arange(T)
    pred = np.argmax(C, axis=1)
    accuracy = float(np.mean(pred == t))
    error = float(np.mean(np.abs(pred - t)) / T)
    diag = np.diag(C)
    rank = float(np.mean(np.mean(C <= diag[:, None], axis=1)))
    return {"accuracy": accuracy, "error": error, "rank": rank}


def summarize(rows):
    d = pd.DataFrame(rows)
    return {
        "mean": float(d["accuracy"].mean()),
        "err": float(d["accuracy"].std(ddof=1) / np.sqrt(len(d))),
        "std": float(d["accuracy"].std(ddof=1)),
        "count": int(len(d)),
        "rank_mean": float(d["rank"].mean()),
        "error_mean": float(d["error"].mean()),
    }


def low_memory_decode(X, seed):
    """Historical repeated 2-fold decoder, one repetition at a time."""
    X = np.asarray(X, dtype=DTYPE)
    N, T, V = X.shape
    rng = np.random.default_rng(seed)
    rows = []

    for rep in range(NREPS):
        fold_ids = get_xval_assignments(N, rng)
        mask0 = (fold_ids == 0)
        mask1 = ~mask0

        # At most two T x V means and one T x T matrix at once.
        A = X[mask0].mean(axis=0, dtype=DTYPE)
        B = X[mask1].mean(axis=0, dtype=DTYPE)
        C = 1.0 - cdist(A, B, metric="correlation")
        C = np.nan_to_num(C, nan=0.0, posinf=0.0, neginf=0.0)

        m0 = decoder_metrics(C); m0.update(rep=rep, fold=0); rows.append(m0)
        m1 = decoder_metrics(C.T); m1.update(rep=rep, fold=1); rows.append(m1)
        del A, B, C

    return summarize(rows)


In [ ]:
def add_archetype_inplace(raw, subjects, k):
    """Add one spatial archetype without allocating an N x T x V temporary."""
    for i, sub in enumerate(subjects):
        t = np.asarray(sub["sXC"], dtype=DTYPE)[:, k]
        v = np.asarray(sub["S"], dtype=DTYPE)[k, :]
        raw[i] += np.multiply.outer(t, v).astype(DTYPE, copy=False)


def normalize_stack_low_memory(raw):
    out = np.empty_like(raw, dtype=DTYPE)
    for i in range(raw.shape[0]):
        X = raw[i]
        mu = X.mean(axis=0, dtype=DTYPE)
        sd = X.std(axis=0, dtype=DTYPE)
        out[i] = (X - mu) / (sd + EPS)
    return out


def full_spatial_stack_low_memory(subjects):
    sxc0 = np.asarray(subjects[0]["sXC"])
    S0 = np.asarray(subjects[0]["S"])
    N, T, V = len(subjects), sxc0.shape[0], S0.shape[1]
    out = np.empty((N, T, V), dtype=DTYPE)

    for i, sub in enumerate(subjects):
        sXC = np.asarray(sub["sXC"], dtype=DTYPE)
        S = np.asarray(sub["S"], dtype=DTYPE)
        X = sXC @ S
        mu = X.mean(axis=0, dtype=DTYPE)
        sd = X.std(axis=0, dtype=DTYPE)
        out[i] = (X - mu) / (sd + EPS)
        del X
    return out


def eval_topm_low_memory(subjects, ranked, top_ms, seed):
    desired = sorted({int(m) for m in top_ms if int(m) <= len(ranked)})
    if not desired:
        return pd.DataFrame()

    # Compute full-model reference, then free it before allocating cumulative raw buffer.
    full = full_spatial_stack_low_memory(subjects)
    fs = low_memory_decode(full, seed)
    N, T, V = full.shape
    chance = 1.0 / T
    del full
    gc.collect()

    raw = np.zeros((N, T, V), dtype=DTYPE)
    rows = []
    wanted = set(desired)

    for pos, k in enumerate(ranked[:max(desired)], start=1):
        add_archetype_inplace(raw, subjects, k)
        if pos not in wanted:
            continue

        recon = normalize_stack_low_memory(raw)
        s = low_memory_decode(recon, seed)
        denom = fs["mean"] - chance

        rows.append({
            "top_m": pos,
            "selected_archetypes": str([int(x) for x in ranked[:pos]]),
            "mean": s["mean"],
            "err": s["err"],
            "std": s["std"],
            "count": s["count"],
            "rank_mean": s["rank_mean"],
            "error_mean": s["error_mean"],
            "full_mean_same_eval": fs["mean"],
            "full_err_same_eval": fs["err"],
            "retention_above_chance": ((s["mean"] - chance) / denom if abs(denom) > 1e-12 else np.nan),
            "chance": chance,
        })
        del recon
        gc.collect()

    del raw
    gc.collect()
    return pd.DataFrame(rows)


In [ ]:
def split_outer(n, seed):
    rng = np.random.default_rng(int(seed))
    idx = np.arange(n)
    rng.shuffle(idx)
    mid = n // 2
    return idx[:mid], idx[mid:]


def ranked_from_rows(df):
    return df.sort_values(
        ["mean_accuracy", "rank_mean", "archetype"],
        ascending=[False, False, True],
    )["archetype"].astype(int).tolist()

KEYS = ["analysis_type","fit_scope","condition","K","outer_rep","direction","top_m"]


## Run/resume the compact spatial grid

Existing completed rows in `msaa_heldout_topm_outputs_spatial_across_fullgrid/` are retained and skipped.


In [ ]:
for K in COMPACT_SPATIAL_K:
    if K not in set(rank_df["K"].astype(int)):
        print(f"SKIP K={K}: no ranking rows", flush=True)
        continue

    print("\n" + "="*80, flush=True)
    print(f"LOW-MEMORY SPATIAL K={K}", flush=True)
    print("="*80, flush=True)

    for condition in ["intact", "word", "rest"]:
        subjects, source = load_condition_subjects(K, condition)
        print(f"{condition} | N={len(subjects)}", flush=True)

        for outer_rep in range(20):
            for direction, offset in [("A_to_B",0), ("B_to_A",1)]:
                r = rank_df[
                    (rank_df["analysis_type"].astype(str) == ANALYSIS_TYPE) &
                    (rank_df["fit_scope"].astype(str) == FIT_SCOPE) &
                    (rank_df["condition"].astype(str) == condition) &
                    (rank_df["K"].astype(int) == K) &
                    (rank_df["outer_rep"].astype(int) == outer_rep) &
                    (rank_df["direction"].astype(str) == direction)
                ].copy()

                if len(r) == 0 or r["archetype"].nunique() != K:
                    print(f"  missing/incomplete ranking outer={outer_rep} {direction}", flush=True)
                    continue

                ranked = ranked_from_rows(r)
                outer_seed = int(r["outer_seed"].iloc[0])
                A, B = split_outer(len(subjects), outer_seed)
                eval_idx = B if direction == "A_to_B" else A

                desired = [m for m in TOP_M_VALUES if m <= K]
                existing = set()
                if len(run_df):
                    cur = run_df[
                        (run_df["condition"].astype(str) == condition) &
                        (run_df["K"].astype(int) == K) &
                        (run_df["outer_rep"].astype(int) == outer_rep) &
                        (run_df["direction"].astype(str) == direction)
                    ]
                    if len(cur):
                        existing = set(cur["top_m"].astype(int))

                missing = [m for m in desired if m not in existing]
                if not missing:
                    continue

                ev = [subjects[int(i)] for i in eval_idx]
                seed = RNG_SEED + 200000*outer_rep + 1000*K + offset
                h = eval_topm_low_memory(ev, ranked, missing, seed)

                h["analysis_type"] = ANALYSIS_TYPE
                h["fit_scope"] = FIT_SCOPE
                h["condition"] = condition
                h["K"] = K
                h["outer_rep"] = outer_rep
                h["direction"] = direction
                h["outer_seed"] = outer_seed
                h["n_evaluation_subjects"] = len(ev)
                h["fit_source_file"] = str(source)
                h["fit_source_basename"] = Path(source).name

                run_df = pd.concat([run_df, h], ignore_index=True) if len(run_df) else h.copy()
                run_df = run_df.drop_duplicates(KEYS, keep="last").reset_index(drop=True)
                run_df.to_csv(RUN_CSV, index=False)

                print(f"  outer {outer_rep+1}/20 {direction} m={missing}", flush=True)
                del ev, h
                gc.collect()

        del subjects
        gc.collect()
        print(f"released K={K} condition={condition}", flush=True)

print("\nLow-memory spatial expansion complete.", flush=True)


In [ ]:
def sem(x):
    x = pd.Series(x).dropna()
    return np.nan if len(x) <= 1 else float(x.std(ddof=1) / np.sqrt(len(x)))

run_df = pd.read_csv(RUN_CSV)
outer = run_df.groupby(
    ["analysis_type","fit_scope","condition","K","top_m","outer_rep"], as_index=False
).agg(
    mean=("mean","mean"),
    full_mean_same_eval=("full_mean_same_eval","mean"),
    retention_above_chance=("retention_above_chance","mean"),
)

summary = outer.groupby(
    ["analysis_type","fit_scope","condition","K","top_m"], as_index=False
).agg(
    mean=("mean","mean"),
    err=("mean",sem),
    std=("mean","std"),
    n_outer=("mean","size"),
    mean_full_same_eval=("full_mean_same_eval","mean"),
    mean_retention=("retention_above_chance","mean"),
    err_retention=("retention_above_chance",sem),
)
summary.to_csv(SUMMARY_CSV, index=False)

coverage = summary[summary["K"].isin(COMPACT_SPATIAL_K)].groupby(["condition","K"]).agg(
    min_outer=("n_outer","min"),
    top_m_values=("top_m", lambda x: sorted(set(map(int,x)))),
).reset_index()

display(coverage)
print("Saved:", SUMMARY_CSV, flush=True)
